In [ ]:
# MUHAMMAD ZAID AQIL BIN ARIZAL (SW01083598) SECTION 01AT
# MUHAMMAD HAZIQ BIN ROHAIZAD (SW01083541) SECTION 01AT

import requests
import time
from bs4 import BeautifulSoup
import csv
# Why need Selenium? because Lazada load their review content with JavaScript, not initial HTML
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [9]:
# Function to scrape reviews from a Lazada review page
# Each review is in div.review-item; name=new-buyer-name, date=new-sku-info, text=LinesEllipsis

def scrape_page(soup, reviews):
    for review_block in soup.find_all('div', class_='review-item'):
        # Reviewer name
        name_el = review_block.find('div', class_='new-buyer-name')
        name = name_el.get_text(strip=True) if name_el else ''
        # Review date
        date_el = review_block.find('div', class_='new-sku-info')
        date = date_el.get_text(strip=True) if date_el else ''
        # Review text (foldable content)
        text_el = review_block.find('div', class_='LinesEllipsis')
        text = text_el.get_text(strip=True) if text_el else ''
        reviews.append({'Text': text, 'Author': name, 'Date': date})

In [3]:
# Base URL and headers
base_url = 'https://www.lazada.com.my/wow/gcp/my/review/product-reviews?itemId=3727981779&skuId=21124212751&spm=a2o4k.pdp_revamp.pdp_top_tab.rating_and_review'
headers = {'User-Agent': 'Mozilla/5.0'}

In [ ]:
# List to store reviews
reviews = []

In [ ]:
# Function to scrape reviews from Lazada (page rendered with JavaScript, use Selenium)
def scrape_all_pages(url):
    driver = webdriver.Chrome()
    try:
        driver.get(url)
        # Wait for Lazada review items to load, then scroll to load more (max 5 loads)
        for _ in range(5): 
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, 'review-item'))
            )
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            time.sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        scrape_page(soup, reviews)
    finally:
        driver.quit()

# 1. Selenium opens a real browser (Chrome).
# 2. The browser loads the page and runs the JavaScript.
# 3. The JavaScript fetches review data (often from an API) and inserts it into the page.
# 4. After that, the DOM contains the div.review-item elements.
# 5. Selenium gives you driver.page_source the fully rendered HTML.
# 6. BeautifulSoup parses that and finds the reviews.


In [ ]:
# Scrape reviews from all pages
scrape_all_pages(base_url)

In [7]:
# Save quotes to CSV file
with open('review-lazada-extracted.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['Text', 'Author', 'Date'])
    writer.writeheader()
    writer.writerows(reviews)